In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from models.linear_probes import linear_probe, linear_probe_tuned
from models.feature_generation import build_feature_bank, extract_encoder, pool_features
from preprocessing.dataset import PipistrelleDataset
import pandas as pd
from evaluation.metrics import plot_comprehensive_boxplots
import pickle
from pathlib import Path
import os
from evaluation.statistical_tests import perform_encoder_statistical_analysis
from preprocessing.dataset import BatAudioPipeline
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import librosa
from pathlib import Path

In [ ]:
# Assuming your BatAudioPipeline class is defined or imported
# from your_module import BatAudioPipeline

def visualize_pipeline_librosa(pipeline_instance, file_path):
    """
    Visualizes the intermediate steps of the BatAudioPipeline 
    using Librosa's superior plotting tools.
    """
    # 1. Extract intermediate states from the pipeline
    audio_raw = pipeline_instance.load(file_path)
    orig_sr = pipeline_instance.orig_sr
    audio_filtered = pipeline_instance.apply_bandpass(audio_raw)
    audio_resampled = pipeline_instance.apply_time_expansion(audio_filtered)
    
    # 2. Convert PyTorch tensors to 1D NumPy arrays for Librosa
    raw_np = audio_raw.squeeze(0).cpu().numpy()
    filtered_np = audio_filtered.squeeze(0).cpu().numpy()
    resampled_np = audio_resampled.squeeze(0).cpu().numpy()
    
    # 3. Setup the Matplotlib figure (3 rows, 2 columns)
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    
    # Helper function to eliminate repetitive plotting boilerplate
    def plot_step(y, sr, row, title, color, n_fft, hop_length):
        # --- Column 1: Librosa Waveshow (Time Domain) ---
        librosa.display.waveshow(y, sr=sr, ax=axes[row, 0], color=color, alpha=0.7)
        axes[row, 0].set_title(f"{title} - Waveform", fontsize=11, fontweight='bold')
        axes[row, 0].set_ylabel("Amplitude")
        axes[row, 0].label_outer() # Hide x-axis labels if shared, clean layout
        
        # --- Column 2: Librosa Specshow (Frequency Domain) ---
        # Compute the Short-Time Fourier Transform (STFT)
        stft_complex = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
        # Convert amplitude to Decibels (dB) relative to peak energy
        stft_db = librosa.amplitude_to_db(np.abs(stft_complex), ref=np.max)
        
        img = librosa.display.specshow(
            stft_db, 
            sr=sr, 
            hop_length=hop_length, 
            x_axis='time', 
            y_axis='linear', 
            ax=axes[row, 1], 
            cmap='magma' # 'magma' or 'inferno' are fantastic for bat sweeps
        )
        axes[row, 1].set_title(f"{title} - Spectrogram (SR: {sr} Hz)", fontsize=11, fontweight='bold')
        fig.colorbar(img, ax=axes[row, 1], format="%+2.0f dB")

    # --- ROW 1: RAW AUDIO (High Ultrasound Sample Rate) ---
    # Higher n_fft used because of the massive sample rate
    plot_step(raw_np, orig_sr, row=0, title="1. Raw Input", color="teal", n_fft=2048, hop_length=512)
    
    # --- ROW 2: FILTERED AUDIO (High Ultrasound Sample Rate) ---
    # Visualizes the highpass/bandreject clean up before downsampling
    plot_step(filtered_np, orig_sr, row=1, title="2. Filtered Signal", color="darkorange", n_fft=2048, hop_length=512)
    
    # --- ROW 3: FINAL AUDIO (Downsampled / Time-Expanded) ---
    # Smaller n_fft used because target sample rate is much lower (e.g., 16kHz)
    plot_step(resampled_np, pipeline_instance.target_sr, row=2, title="3. Resampled (Perch Ready)", color="crimson", n_fft=512, hop_length=128)
    
    # Add axis labels to the bottom-most plots explicitly
    axes[2, 0].set_xlabel("Time (seconds)")
    axes[2, 1].set_xlabel("Time (seconds)")
    
    plt.tight_layout()
    plt.show()


In [ ]:
pipeline = BatAudioPipeline(
        target_sr=16000, 
        expansion_factor=10, 
        filter_echo=False, # Set to True to see the bandreject filter in action
        window_sec=10, 
    )
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "data" / "xenocanto-dataset")
# Replace with a path to one of your ultrasound bat .wav files
file_path = path + "\\type C\XC938943 - Common Pipistrelle - Pipistrellus pipistrellus.wav"
# Run the visualization
visualize_pipeline_librosa(pipeline, file_path)

In [ ]:
pipeline = BatAudioPipeline(
        target_sr=16000, 
        expansion_factor=10, 
        filter_echo=True, # Set to True to see the bandreject filter in action
        window_sec=10
    )
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "data" / "xenocanto-dataset")
# Replace with a path to one of your ultrasound bat .wav files
file_path = path + "\\type C\XC938943 - Common Pipistrelle - Pipistrellus pipistrellus.wav"
# Run the visualization
visualize_pipeline_librosa(pipeline, file_path)

In [ ]:
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = "effnetb0",
                             filter_echo = False)

In [ ]:
print(batdata[0][0][0].shape)

In [ ]:
def visualize_pipeline_librosa_win(pipeline_instance, file_path):
    """
    Visualizes the pipeline steps PLUS the first 3 windows.
    """
    # 1. Pipeline Execution
    audio_raw = pipeline_instance.load(file_path)
    orig_sr = pipeline_instance.orig_sr
    audio_filtered = pipeline_instance.apply_bandpass(audio_raw)
    audio_resampled = pipeline_instance.apply_time_expansion(audio_filtered)
    
    # Generate windows
    # Note: window_audio expects [1, Time], output is [Num_Windows, Win_Samples]
    windows = pipeline_instance.window_audio(audio_resampled) 
    
    # 2. Setup Figure (3 rows for pipeline + 3 rows for windows = 6 rows)
    fig, axes = plt.subplots(6, 2, figsize=(15, 20))
    
    # Helper to plot on axes[row, col]
    def plot_step(y, sr, row, title, color, n_fft, hop_length, is_window=False):
        # Waveform
        librosa.display.waveshow(y, sr=sr, ax=axes[row, 0], color=color, alpha=0.7)
        axes[row, 0].set_title(f"{title} - Waveform", fontsize=10)
        
        # Spectrogram
        stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
        db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
        librosa.display.specshow(db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=axes[row, 1], cmap='magma')
        axes[row, 1].set_title(f"{title} - Spectrogram", fontsize=10)

    # --- PIPELINE STEPS (Rows 0-2) ---
    raw_np = audio_raw.squeeze(0).cpu().numpy()
    filtered_np = audio_filtered.squeeze(0).cpu().numpy()
    resampled_np = audio_resampled.squeeze(0).cpu().numpy()
    
    plot_step(raw_np, orig_sr, 0, "1. Raw", "teal", 2048, 512)
    plot_step(filtered_np, orig_sr, 1, "2. Filtered", "darkorange", 2048, 512)
    plot_step(resampled_np, pipeline_instance.target_sr, 2, "3. Resampled", "crimson", 512, 128)
    
    # --- WINDOWS (Rows 3-5) ---
    for i in range(3):
        if i < windows.shape[0]:
            win_np = windows[i].cpu().numpy()
            plot_step(win_np, pipeline_instance.target_sr, i+3, f"Window {i+1}", "purple", 512, 128)
        else:
            axes[i+3, 0].text(0.5, 0.5, "No window available", ha='center')
            axes[i+3, 1].text(0.5, 0.5, "No window available", ha='center')

    plt.tight_layout()
    plt.show()

In [ ]:
pipeline = BatAudioPipeline(
        target_sr=16000, 
        expansion_factor=10, 
        filter_echo=False, # Set to True to see the bandreject filter in action
        window_sec=10, 
    )
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "data" / "xenocanto-dataset")
# Replace with a path to one of your ultrasound bat .wav files
file_path = path + "\\type C\XC938943 - Common Pipistrelle - Pipistrellus pipistrellus.wav"
# Run the visualization
visualize_pipeline_librosa_win(pipeline, file_path)

In [ ]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
X_per2 = np.load(path + "\\perch2-2.npz")['pooled_feats']
X_eff0 = np.load(path + "\\AVEX-effnetb0-2.npz")['pooled_feats']
X_NLM = np.load(path + "\\NLM_BEATs-2.npz")['pooled_feats']
y =  np.load(path + "\\NLM_BEATs-2.npz")['labels']
label_names = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']

In [ ]:
X_eff0_pooled = pool_features(X_eff0, windows=False, window_pooled=True, method='mean',encoder="effnetb0")
X_NLM_pooled = pool_features(X_NLM, windows=False, window_pooled=True, method='mean',encoder="NLM_BEATs")
X_per2_pooled = pool_features(X_per2, windows=False, window_pooled=True, method='mean',encoder="perch2")

In [ ]:
eff0_all_results = linear_probe_tuned(X_eff0_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
per2_all_results =linear_probe_tuned(X_per2_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
NLM_all_results =linear_probe_tuned(X_NLM_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:

results_test = {
            'ESP-EffNetB0': eff0_all_results,
            'Perch 2.0': per2_all_results,
            'NatureLM-BEATs': NLM_all_results,
        }

plot_comprehensive_boxplots(results_test, label_names=label_names)

In [ ]:
df_stat = perform_encoder_statistical_analysis(results_test, label_names=label_names)

In [ ]:
from evaluation.statistical_tests import perform_and_plot_nemenyi
perform_and_plot_nemenyi(results_test, label_names=label_names)

In [ ]:
from evaluation.statistical_tests import evaluate_and_plot_linear_probe_algorithms
evaluate_and_plot_linear_probe_algorithms(per2_all_results, label_names=label_names)

In [ ]:
from evaluation.tables import process_encoder_folds_to_table
latex_code = process_encoder_folds_to_table(results_test)
print(latex_code)

In [ ]:
with open("encoder_table.tex", "w", encoding="utf-8") as f:
    f.write(latex_code)

In [ ]:
from evaluation.metrics import plot_comprehensive_calibration,plot_mlp_balancing_boxplots,plot_comprehensive_mlp_boxplots
from models.MLP_balancing import balancing_mlp,balancing_mlp_val,data_augmented_mlp
import pickle
from pathlib import Path
import os
from evaluation.statistical_tests import perform_encoder_statistical_analysis,evaluate_and_plot_mlp_strategies

In [ ]:
balancing_results =balancing_mlp_val(X_per2_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
plot_comprehensive_mlp_boxplots(balancing_results, augmentation_results, label_names=label_names)

In [ ]:
from evaluation.tables import  process_folds_to_table
latex_code = process_folds_to_table(balancing_results, augmentation_results)
with open("balancing_table.tex", "w", encoding="utf-8") as f:
    f.write(latex_code)

In [ ]:
evaluate_and_plot_mlp_strategies(balancing_results,augmentation_results, label_names=label_names)

In [ ]:
#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "perch2"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = False)

X_bags,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)

In [ ]:
pickle_filename = 'perch2-bags.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(X_bags, pickle_file)

In [ ]:
X_bags_pooled = pool_features(X_bags, windows  = False,window_pooled  = False, method  ='mean',encoder  = 'perch2')

In [ ]:
from models.abmil_model import abmil_classifier_tuned
abmil_results = abmil_classifier_tuned(X_bags_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42,
                           n_iter_search = 4,label_specific_features = False,hybrid = False,
                           ensemble = True,ensemble_labels = [0])

dir = Path(os.getcwd()).resolve().parent 
root_dir = str(dir / "data" / "xenocanto-dataset")
csv_data = str(dir / "data" / "bat_metadata.csv")
augmentation_results = data_augmented_mlp(csv_data,root_dir,X_per2_pooled,y, n_split=5, num_trials=5,random_state=42)

#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "perch2"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = False)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'perch2-2-noecho.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "NLM_BEATs"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = False)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'NLM_BEATs-2-noecho.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "effnetb0"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = False)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'AVEX-effnetb0-2-noecho.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

In [ ]:
pickle_filename = 'augment_results.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(augmentation_results, pickle_file)

In [ ]:
pickle_filename = 'abmil_results-2.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(abmil_results, pickle_file)

In [ ]:

#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "perch2"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = True)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'perch2-2-noecho.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "NLM_BEATs"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = True)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'NLM_BEATs-2-noecho.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "effnetb0"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = True)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'AVEX-effnetb0-2-noecho.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

In [ ]:
from evaluation.metrics import plot_abmil_vs_perch_boxplots
plot_abmil_vs_perch_boxplots(abmil_results, per2_all_results, label_names=label_names)

In [ ]:
from evaluation.tables import process_abmil_vs_perch_table
lat_code = process_abmil_vs_perch_table(abmil_results, per2_all_results)
with open("abmil_table.tex", "w", encoding="utf-8") as f:
    f.write(lat_code)

In [ ]:
from evaluation.statistical_tests import evaluate_abmil_vs_perch_statistical
evaluate_abmil_vs_perch_statistical(abmil_results, per2_all_results, label_names=None)

In [ ]:
from evaluation.statistical_tests import evaluate_and_plot_model_comparison_abmil
evaluate_and_plot_model_comparison_abmil(per2_all_results,abmil_results,label_names = label_names)

In [ ]:
from evaluation.metrics import plot_comprehensive_calibration2
plot_comprehensive_calibration2(per2_all_results, abmil_results, label_names, n_bins=10, strategy="uniform")

In [ ]:
from evaluation.tables import generate_abmil_calibration_latex
lat_code = generate_abmil_calibration_latex(abmil_results, label_names, n_bins=10, strategy="uniform")
with open("calibration_table.tex", "w", encoding="utf-8") as f:
    f.write(lat_code)

In [ ]:
print(abmil_results[0]['y_true_cv'])
print(abmil_results[0]['oof_indices'])

In [ ]:
print(abmil_results[0]['oof_indices'][56]) #corresponds to last fold 1 test (+ 2 on output to find inline)
print(abmil_results[0]['oof_indices'][79])
print(abmil_results[0]['oof_indices'][98])
print(abmil_results[0]['oof_indices'][109])

In [ ]:
target_indices = [abmil_results[0]['oof_indices'][57],
                  abmil_results[0]['oof_indices'][79],
                  abmil_results[0]['oof_indices'][98],
                  abmil_results[0]['oof_indices'][109]]

In [ ]:
from evaluation.attention_visual import plot_abmil_attention_on_pure_recordings
from models.abmil_model import predict_abmil
plot_abmil_attention_on_pure_recordings(abmil_results, X_bags_pooled, y, target_indices, label_names,predict_abmil_fn=predict_abmil)

In [ ]:
import os
from evaluation.attention_visual import plot_flashy_abmil_waveforms
dir = Path(os.getcwd()).resolve().parent / "data"
plot_flashy_abmil_waveforms(abmil_results = abmil_results, X_bags = X_bags_pooled, y_true = y, target_indices =target_indices,
                            class_names = label_names,data_input_csv=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),predict_abmil_fn=predict_abmil)

In [ ]:
from evaluation.attention_visual import plot_flashy_abmil_spectrograms
dir = Path(os.getcwd()).resolve().parent / "data"
plot_flashy_abmil_spectrograms(abmil_results = abmil_results, X_bags = X_bags_pooled, y_true = y, target_indices =target_indices,
                            class_names = label_names,data_input_csv=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),predict_abmil_fn=predict_abmil)

In [ ]:
from models.abmil_model import analyze_best_hyperparameters
# Execute the analysis
top_configs = analyze_best_hyperparameters(abmil_results)

In [ ]:
from models.abmil_model import abmil_classifier_deployement
ab_deployement_results = abmil_classifier_deployement(X_bags_pooled, y, n_split=5, random_state=42)

In [ ]:
# 2. Extract and print the best configurations
for result in ab_deployement_results:
    print("=" * 50)
    print(f" MODEL: {result['model']} CONFIGURATION")
    print("=" * 50)
    print(f"Best Inner CV Score (Macro AP): {result['best_score']:.4f}\n")
    
    print("Winning Hyperparameters:")
    for param_name, param_value in result['best_params'].items():
        print(f"  ✦ {param_name}: {param_value}")
    print("=" * 50)

In [ ]:
print(augmentation_results)